# Speaker embeddings with ESPnet-SPK: switching models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/spk_models_demo.ipynb) [![spk_models_demo](https://github.com/espnet/notebook/actions/workflows/spk_models_demo.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/spk_models_demo.yml)

Every ESPnet speaker model on Hugging Face loads through the same call, so
changing the embedding is a one-line change. This notebook puts the released
VoxCeleb models behind one selector, extracts embeddings from your own files,
and compares the models on the same recordings.

The mel and raw-waveform models run on CPU. The WavLM models download
1.4–2.1 GB, need the extra install below, and are much faster on a GPU.

## Install

In [ ]:
%pip install -q "espnet[spk]==202610.post1" espnet_model_zoo librosa
# The WavLM models load their front end through S3PRL, which the 202610.post1
# spk extra does not pull in yet. Uncomment for those models:
# %pip install -q espnet-s3prl

## The models

All are trained on VoxCeleb1+2 (the `voxcelebs12devs_*` ones add more data) and
released under CC BY 4.0. EER is on VoxCeleb1-O, as reported on each model card;
lower is better. Size is the checkpoint download.

| Model | Encoder | Front end | EER (%) | Size |
|---|---|---|---|---|
| [`voxcelebs12_rawnet3`](https://huggingface.co/espnet/voxcelebs12_rawnet3) | RawNet3 | raw waveform | 0.739 | 114 MB |
| [`voxcelebs12_ecapa_mel`](https://huggingface.co/espnet/voxcelebs12_ecapa_mel) | ECAPA-TDNN | mel | 0.856 | 109 MB |
| [`voxcelebs12_ska_mel`](https://huggingface.co/espnet/voxcelebs12_ska_mel) | SKA-TDNN | mel | 0.729 | 193 MB |
| [`voxcelebs12_mfaconformer_mel`](https://huggingface.co/espnet/voxcelebs12_mfaconformer_mel) | MFA-Conformer | mel | 0.782 | 261 MB |
| [`voxcelebs12_xvector_mel`](https://huggingface.co/espnet/voxcelebs12_xvector_mel) | x-vector | mel | 1.814 | 151 MB |
| [`voxcelebs12_ecapa_frozen`](https://huggingface.co/espnet/voxcelebs12_ecapa_frozen) | ECAPA-TDNN | WavLM, frozen | 0.638 | 1.4 GB |
| [`voxcelebs12_ecapa_wavlm_joint`](https://huggingface.co/espnet/voxcelebs12_ecapa_wavlm_joint) | ECAPA-TDNN | WavLM, fine-tuned | 0.394 | 1.4 GB |
| [`voxcelebs12_ska_wavlm_frozen`](https://huggingface.co/espnet/voxcelebs12_ska_wavlm_frozen) | SKA-TDNN | WavLM, frozen | 0.564 | 2.1 GB |
| [`voxcelebs12_ska_wavlm_joint`](https://huggingface.co/espnet/voxcelebs12_ska_wavlm_joint) | SKA-TDNN | WavLM, fine-tuned | 0.516 | 2.1 GB |
| [`voxcelebs12devs_voxblinkclean_rawnet3`](https://huggingface.co/espnet/voxcelebs12devs_voxblinkclean_rawnet3) | RawNet3 | raw waveform, + VoxBlink clean | 0.771 | 241 MB |
| [`voxcelebs12devs_voxblinkfull_rawnet3`](https://huggingface.co/espnet/voxcelebs12devs_voxblinkfull_rawnet3) | RawNet3 | raw waveform, + VoxBlink full | 0.782 | 377 MB |
| [`voxcelebs12devs_librispeech_cv16fa_rawnet3`](https://huggingface.co/espnet/voxcelebs12devs_librispeech_cv16fa_rawnet3) | RawNet3 | raw waveform, + LibriSpeech, CV16 | 1.229 | 151 MB |

## Pick a model

In Colab this is a drop-down. Anywhere else, edit the string. Any other ESPnet
speaker model id on Hugging Face works too.

In [ ]:
MODEL = "espnet/voxcelebs12_rawnet3"  # @param ["espnet/voxcelebs12_rawnet3", "espnet/voxcelebs12_ecapa_mel", "espnet/voxcelebs12_ska_mel", "espnet/voxcelebs12_mfaconformer_mel", "espnet/voxcelebs12_xvector_mel", "espnet/voxcelebs12_ecapa_frozen", "espnet/voxcelebs12_ecapa_wavlm_joint", "espnet/voxcelebs12_ska_wavlm_frozen", "espnet/voxcelebs12_ska_wavlm_joint", "espnet/voxcelebs12devs_voxblinkclean_rawnet3", "espnet/voxcelebs12devs_voxblinkfull_rawnet3", "espnet/voxcelebs12devs_librispeech_cv16fa_rawnet3"] {allow-input: true}

## Loading and embedding

`load` keeps every model it has built, so switching back costs nothing.
`embed` takes a path or a 16 kHz array and returns one vector.

In [ ]:
import librosa
import numpy as np
import torch
from espnet2.bin.spk_inference import Speech2Embedding

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FS = 16000  # every model above expects 16 kHz
_models = {}


def load(model_id):
    if model_id not in _models:
        _models[model_id] = Speech2Embedding.from_pretrained(model_id, device=DEVICE)
    return _models[model_id]


def embed(audio, model_id=None):
    if isinstance(audio, str):
        audio, _ = librosa.load(audio, sr=FS)
    with torch.no_grad():
        emb = load(model_id or MODEL)(audio)
    return emb[0].cpu().numpy()


def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


print("device:", DEVICE)

## Two recordings

Two different sentences, read by two different people. Replace them with your
own files: any format librosa reads, any sample rate (it is resampled to 16 kHz).

In [ ]:
from IPython.display import Audio, display

!wget -q -O a.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
!wget -q -O b.wav https://github.com/espnet/espnet/raw/master/test_utils/st_test.wav

for name in ("a.wav", "b.wav"):
    audio, _ = librosa.load(name, sr=FS)
    print(name, f"{len(audio) / FS:.1f} s")
    display(Audio(audio, rate=FS))

## One model

The score is the cosine between two embeddings. "Same speaker" cuts one
recording in two; "different speakers" compares the two files. Where to put
the decision threshold depends on the model and on your data, so pick it on
recordings like yours rather than copying a number.

In [ ]:
a, _ = librosa.load("a.wav", sr=FS)
half = len(a) // 2

first, second = embed("a.wav"), embed("b.wav")
same = cosine(embed(a[:half]), embed(a[half:]))
different = cosine(first, second)

print(MODEL)
print("embedding size:     ", first.shape)
print(f"same speaker        {same:.3f}")
print(f"different speakers  {different:.3f}")

## Comparing models

Runs the same test on several models. The default list is the CPU-friendly
ones; add the WavLM models when you have a GPU. Cosine scores are not
comparable across models in absolute terms: look at the gap between the two
columns for each model.

In [ ]:
import time

COMPARE = [
    "espnet/voxcelebs12_rawnet3",
    "espnet/voxcelebs12_ecapa_mel",
    "espnet/voxcelebs12_ska_mel",
   
    "espnet/voxcelebs12_xvector_mel",
    # "espnet/voxcelebs12_ecapa_wavlm_joint",  # 1.4 GB, best EER; use a GPU
]

b, _ = librosa.load("b.wav", sr=FS)
print(f"{'model':45s} {'dim':>5s} {'same':>7s} {'diff':>7s} {'gap':>7s} {'sec':>6s}")
for model_id in COMPARE:
    load(model_id)  # download and build outside the timing
    start = time.time()
    ea, eb = embed(a, model_id), embed(b, model_id)
    s = cosine(embed(a[:half], model_id), embed(a[half:], model_id))
    d = cosine(ea, eb)
    sec = time.time() - start
    print(f"{model_id:45s} {ea.shape[0]:5d} {s:7.3f} {d:7.3f} {s - d:7.3f} {sec:6.2f}")

## Your own files

Embeds every audio file in a folder with the selected model and saves the
matrix plus the file order, ready for clustering, plotting or as a speaker
conditioning input.

In [ ]:
from pathlib import Path

AUDIO_DIR = "."  # @param {type:"string"}
EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}

files = sorted(p for p in Path(AUDIO_DIR).iterdir() if p.suffix.lower() in EXTENSIONS)
embeddings = np.stack([embed(str(p)) for p in files])

out = f"embeddings_{MODEL.split('/')[-1]}"
np.save(out + ".npy", embeddings)
Path(out + ".txt").write_text("\n".join(str(p) for p in files) + "\n")
print(embeddings.shape, "->", out + ".npy")

scores = embeddings @ embeddings.T
norms = np.linalg.norm(embeddings, axis=1)
print(np.round(scores / np.outer(norms, norms), 3))

## Where next

- **Training your own**: the recipe behind these models is
  [`egs2/voxceleb/spk1`](https://github.com/espnet/espnet/tree/master/egs2/voxceleb/spk1)
- **The paper**: Jung et al., "ESPnet-SPK: full pipeline speaker embedding
  toolkit with reproducible recipes, self-supervised front-ends, and
  off-the-shelf models", Interspeech 2024
- **One model, step by step**: [`spk_demo.ipynb`](spk_demo.ipynb)